[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/internshipinabook/mlops-internship-in-a-book/blob/main/notebooks/week1_version_everything_STARTER.ipynb)


[![Get the Book on Selar](https://img.shields.io/badge/Get%20the%20full%20book-Selar-1E7A34?style=for-the-badge)](https://selar.com/7m27877571)

# Week 1 -- Version Everything (STARTER)
### The MLOps Internship | DataFlow Infrastructure

**Dataset:** CreditDefault-NG (UCI Credit Card Default)

**This week:** DVC data versioning, MLflow experiment tracking, model registry, reproducible training runs

STARTER -- complete each exercise before moving on.


In [ ]:
# -- Colab Setup (skip if running locally) -------------------------
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/mlops-internship-in-a-book.git mlops-book
    %cd mlops-book
    !pip install -r requirements.txt -q
os.makedirs('outputs', exist_ok=True)
os.makedirs('models',  exist_ok=True)
os.makedirs('datasets', exist_ok=True)
print('Setup complete.')


In [ ]:
# -- Reproducibility Seeds ------------------------------------------
# SEED=42 ensures every random operation gives the same result on every run.
# Set seeds BEFORE any data loading, model creation, or training.
import random, numpy as np
SEED = 42
random.seed(SEED)       # Python random
np.random.seed(SEED)    # NumPy random (used by sklearn)
print(f'Seeds fixed: {SEED}')


## Learning Objectives

- Version the CreditDefault-NG dataset using DVC and commit the manifest to git
- Track a training run in MLflow: parameters, metrics, and the model artifact
- Reproduce a past experiment exactly from a DVC commit and MLflow run ID
- Build a model registry entry with stage transitions: Staging, Production, Archived
- Write the experiment log that lets someone reproduce your Week 1 run in six months



---

## MONDAY -- "DVC: Versioning the Data"


DVC tracks large data files without storing them in git. It stores a small .dvc file -- a hash of the data -- in git, and the actual data in remote storage. When you run dvc pull, DVC downloads the exact version of the data corresponding to the current git commit. Check out any past commit and you get the data that existed at that point in time. This is how you reproduce the January model: check out January's commit, DVC pulls January's data, and you train from there.


### Exercise 1.1 -- DVC init and track

Initialise DVC in the repository. Add data/credit_default_ng.csv. Commit the .dvc file. Push to remote. Then delete the local CSV and run dvc pull to verify exact restoration. Confirm the md5 hash in the .dvc file matches.


In [ ]:
# Exercise 1.1: DVC init and track
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


### Exercise 1.2 -- Data hash verification

Read the md5 hash from the .dvc file. Modify one row of the CSV and run dvc status. What does DVC report? Restore the original and verify the hash matches.


In [ ]:
# Exercise 1.2: Data hash verification
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Monday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Monday: DVC: Versioning the Data --
dvc init                                              # creates .dvc/ -- DVC's internal config, separate from .git/
dvc add data/credit_default_ng.csv                    # hashes the file, writes credit_default_ng.csv.dvc, moves data into DVC cache
git add data/credit_default_ng.csv.dvc .gitignore     # only the small .dvc pointer file goes into git -- never the raw data
git commit -m 'Track CreditDefault-NG dataset with DVC'
dvc remote add -d s3remote s3://dataflow-dvc/credit-model   # -d marks this the default remote for push/pull
dvc push                                              # uploads the actual data to remote storage (git never sees it)


### Expected Output (illustrative)

```
$ dvc init
Initialized DVC repository.

$ dvc add data/credit_default_ng.csv
100% Adding...|████████████████████████████████████████| 1/1
To track the changes with git, run:
    git add data/credit_default_ng.csv.dvc data/.gitignore

$ git commit -m 'Track CreditDefault-NG dataset with DVC'
[main 4a1f2c9] Track CreditDefault-NG dataset with DVC
 2 files changed, 5 insertions(+)

$ dvc push
1 file pushed
```
**Read the `.dvc` file it created** — it's just a hash and a size, nothing else:
```yaml
outs:
- md5: 8f14e45fceea167a5a36dedd4bea2543
  size: 3847201
  path: credit_default_ng.csv
```
That `md5` value is what makes the dataset reproducible: anyone with this `.dvc` file and
access to the remote can `dvc pull` the *exact same bytes*, forever.


### Monday Morning Moment

*Slack -- Monday, 10:00am.*

**Sola Fashola:** Did you check the DVC setup before touching the dataset?

**Temi Adeyemi:** I was going to start with the training script.

**Sola Fashola:** DVC first. Always. If you train without versioning the data, you produce a model you cannot reproduce. The training script will still be there in twenty minutes.

**Temi Adeyemi:** What happens if I forget to run dvc add before committing?

**Sola Fashola:** Then your git commit references a dataset no future checkout can retrieve. The .dvc file is missing. You have committed a pointer to nothing.

**Dr. Emeka Obi:** We lost three weeks of training runs last year for exactly this reason. DVC first.




---

## TUESDAY -- "MLflow: Tracking Experiments"


MLflow tracks three things for every training run: parameters (what you set before training), metrics (what the model achieved), and artifacts (the files the run produced, including the model itself). Every run gets a unique run_id. You can compare runs in the MLflow UI, reproduce any run from its ID, and promote runs to the model registry. The alternative -- a spreadsheet, a folder of model files, a Slack message saying 'the Tuesday run was better' -- is how the January model became unreproducible.


### Exercise 1.3 -- MLflow tracking run

Train the credit default model. Log: at minimum 3 parameters, 3 metrics, and the model artifact. Open the MLflow UI (mlflow ui, then http://localhost:5000) and verify all logged values appear correctly.


In [ ]:
# Exercise 1.3: MLflow tracking run
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Tuesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Tuesday: MLflow: Tracking Experiments --
import mlflow, mlflow.sklearn
mlflow.set_experiment('credit-default-ng')             # creates the experiment on first call, reuses it after

with mlflow.start_run() as run:                        # run.info.run_id is generated here -- this IS the reproducibility key
    # Log what you set
    mlflow.log_param('model_type', 'RandomForestClassifier')
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_depth', 8)
    # ... train the model here (fit on X_train, y_train) ...
    # Log what the model achieved
    mlflow.log_metric('val_auc', val_auc)              # val_auc computed from your held-out validation split
    mlflow.log_metric('val_accuracy', val_acc)
    # Log the artifact itself, so 'the model' and 'this run' can never drift apart
    mlflow.sklearn.log_model(model, 'credit_default_model')

print(f"Run ID: {run.info.run_id}")                    # you will need this exact string on Thursday


### Expected Output (illustrative)

```
Run ID: cc8a3f2d91b74e0a9f7c3e5b6a1d8f42
```
Opening `mlflow ui` (`http://localhost:5000`) and clicking into this run shows three panels:

| Parameters | Metrics | Artifacts |
|---|---|---|
| model_type: RandomForestClassifier | val_auc: 0.8814 | credit_default_model/ (MLmodel, model.pkl, conda.yaml) |
| n_estimators: 100 | val_accuracy: 0.812 | |
| max_depth: 8 | | |

**Why this matters:** if `val_auc` were logged without `n_estimators` and `max_depth` next to
it, 0.8814 would be a number with no way to reproduce it. The three log calls above are what
make the metric *meaningful*, not just recorded.



---

## WEDNESDAY -- "The Model Registry"


The MLflow Model Registry is the single source of truth for which model version is in Production. You never deploy a model that is not in the registry. You never promote a model from Staging to Production without a passing evaluation suite. Every stage transition is logged with a timestamp and a user. When Farida asks in six months 'which model made this decision?', the registry has the answer.


**Wednesday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Wednesday: The Model Registry --
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Register the model from this run
model_uri = f'runs:/{run.info.run_id}/credit_default_model'
result = mlflow.register_model(model_uri, 'credit-default-ng')
print(f'Version {result.version} registered')

# Promote to Staging -- evaluation required before Production
client.transition_model_version_stage(
    name='credit-default-ng',
    version=result.version,
    stage='Staging',
)
print('In Staging. val_AUC must exceed 0.80 before Production.')


### Expected Output (illustrative)

```
Version 1 registered in stage 'None'
```
A freshly registered model always starts in stage `None` — promoting it to `Staging` or
`Production` is a deliberate, separate action (see Week 7's Staging-vs-Production exercise),
never automatic on registration.



---

## THURSDAY -- "Reproducing a Past Experiment"


The test of any versioning system is whether you can reproduce a past experiment exactly. Not approximately -- exactly. Today: reproduce the model that was in production in January using the January git commit and data version. The val_AUC should match to four decimal places. If it does not, find out why: wrong seed, different library version, or -- most likely -- the data was not versioned at the time.

Pause and Predict -- before checking: do you think the January model is reproducible? What information would you need to reproduce it? Write your prediction before running the reproduction attempt.


### STOP AND TRACE

Before running:

import mlflow
run = mlflow.get_run('your_run_id_here')
params  = run.data.params
metrics = run.data.metrics
artifact_uri = run.info.artifact_uri

What type does run.data.params return? What type does run.data.metrics return?
What is in run.info vs run.data?
Why this matters: to reproduce a run you need run.data.params. To verify reproduction you need run.data.metrics. To reload the model you need run.info.artifact_uri. These are stored in three separate places.


In [ ]:
# Exercise 1.4: STOP AND TRACE -- MLflow run retrieval
# ----------------------------------------------------------
# Hint: read the markdown cell above carefully.
# Hint: check the Common Mistakes section at the end.

# YOUR CODE HERE


**Thursday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Thursday: Reproducing a Past Experiment --
# Checkout the January state of the repository
git checkout a3f8c91  # January commit hash (from git log)
dvc pull              # downloads the January dataset version

# Reproduce the January training run
python train.py --config configs/jan_config.yaml

# Compare with the logged MLflow run
import mlflow
jan_run = mlflow.get_run('jan_run_id_here')
print(f'January AUC: {jan_run.data.metrics["val_auc"]}')
print(f'Reproduced:  {val_auc}')
# Should match to 4 decimal places if SEED=42 is set correctly


### Expected Output (illustrative)

```
val_AUC 0.8814
```
matched to four decimal places, exactly as the Friday Workplace Moment states. If your
reproduced value differs even in the fourth decimal place, something in the recorded
provenance (git commit, DVC hash, or hyperparameters) doesn't actually match what you
retrieved — treat any mismatch as a bug to find, not rounding noise to ignore.



---

## FRIDAY -- "The Friday Build: Experiment Log"


Write the experiment log that makes your Week 1 training run reproducible for anyone on the team in six months. It must contain: the git commit hash, the DVC data version hash (from the .dvc file), the MLflow run ID, the hyperparameters, the validation metrics, the Python and library versions, and a one-paragraph note on what you tried and why. The note is the most important part -- it tells the next person why you made the decisions you made.


**Friday scaffold** (read and run, then adapt in exercises):


In [ ]:
# -- Friday: The Friday Build: Experiment Log --
# experiment_log.md template:
# Run ID: [mlflow run id from run.info.run_id]
# Git commit: [git rev-parse HEAD]
# DVC data hash: [md5 from data/credit_default_ng.csv.dvc]
# Date: 2026-06-08
# Python: 3.10.12  MLflow: 2.12.0  scikit-learn: 1.4.2
# Hyperparameters:
#   n_estimators: 100
#   max_depth: 8
#   random_state: 42
# Metrics: val_AUC=0.XXXX  val_F1_weighted=0.XXXX
# Notes: Baseline run. Tried max_depth 6 and 10 -- 8 was optimal on val set.


### Expected Output (illustrative)

The experiment log this cell scaffolds should read like:
```
Run ID: cc8a3f2d91b74e0a9f7c3e5b6a1d8f42
Git commit: e4b7a91
DVC data hash: 8f14e45fceea167a5a36dedd4bea2543
Date: 2026-06-08
Python: 3.10.12  MLflow: 2.12.0  scikit-learn: 1.4.2
Hyperparameters: n_estimators=100, max_depth=8
val_AUC: 0.8814
```
Six months from now, someone with only this text block and repo access should be able to
reproduce your exact result — that's the actual test of whether Week 1 succeeded.


### Friday Workplace Moment

*DataFlow Infrastructure -- Friday standup.*

**Dr. Emeka Obi:** Can you reproduce this week's best model from the run ID alone?

**Temi Adeyemi:** Run ID cc8a3f2d. Git checkout e4b7a91. DVC pull. Python train.py with the logged hyperparameters. val_AUC 0.8814. Matched to four decimal places.

**Sola Fashola:** The January production model. Can you reproduce that?

**Temi Adeyemi:** No. The January run was not logged in MLflow. We have the model artifact but not the training data version or the hyperparameters.

**Dr. Emeka Obi:** That gap is what caused this incident. The model existed. Its history did not. Write that in the postmortem.

**Sola Fashola:** From this week forward: every model that goes to Production must have a logged MLflow run and a DVC-tracked dataset. No exceptions.



---

## 🎁 BONUS EXERCISE — Resolving a Merge Conflict in a `.dvc` File

*Not required for the core path. Recommended if you want practice with the Git workflows real
MLOps teams use daily.*

**Scenario:** You and a teammate both update `data/credit_default_ng.csv` on separate branches
before either of you pulls the other's change. Because DVC only commits a small pointer file
to git, the conflict shows up as a conflict in the `.dvc` file itself — not in the data.

```bash
# Simulate it:
git checkout -b feature/add-q2-data
dvc add data/credit_default_ng.csv   # (after appending Q2 rows locally)
git add data/credit_default_ng.csv.dvc
git commit -m "Add Q2 data"

git checkout main
git checkout -b feature/fix-nulls
dvc add data/credit_default_ng.csv   # (after a different local edit: null cleanup)
git add data/credit_default_ng.csv.dvc
git commit -m "Fix null values in training data"

git checkout main
git merge feature/add-q2-data
git merge feature/fix-nulls   # <- conflicts here
```

**Your task:**
1. Read the conflict markers in `credit_default_ng.csv.dvc` — notice it's just two different
   `md5` hashes competing, not a line-by-line text diff.
2. Decide: do you need *both* changes combined into one dataset, or does one supersede the
   other? (Hint: in this scenario, you need both — Q2 rows AND null cleanup.)
3. Resolve by re-running the transformation that combines both changes on top of one branch,
   then `dvc add` again to produce a single new hash that supersedes both conflicting ones.
4. Commit the resolution and confirm `dvc status` reports clean.

**Why this matters:** a `.dvc` merge conflict can *look* resolved (git shows no conflict
markers) while actually pointing to data that has neither fix applied. Always verify by
running `dvc pull` and checking the data itself after resolving — never trust the git merge
alone.


## YOUR WEEK 1 CHECKLIST

Check each item before moving on.

- [ ] The CreditDefault-NG dataset is tracked with DVC and the .dvc file is committed to git.
- [ ] I can reproduce any MLflow run from its run_id, git commit hash, and DVC data hash.
- [ ] The model registry has at least one version in Staging with logged evaluation metrics.
- [ ] I know the difference between mlflow.log_param, mlflow.log_metric, and mlflow.log_artifact.
- [ ] My experiment log contains everything needed to reproduce this run in six months.


## Challenge Task

> **Core path:** Build a git pre-commit hook that fails the commit if any .csv or .parquet file is staged without a corresponding .dvc file. This makes DVC-first automatic rather than a manual discipline.

> **Advanced path:** Implement an MLflow comparison script that takes two run IDs and produces a markdown table: parameters that differ, metrics that differ, and which run wins on each metric. Useful for the validation gate in Week 5.


## Common Mistakes

**Training before DVC init:** A model trained before the dataset is DVC-tracked has no reproducible data provenance. You can log the model in MLflow but you cannot reproduce its training data.

**Logging metrics without parameters:** val_AUC=0.88 means nothing without knowing n_estimators, max_depth, and the random seed. Log both, always.

**Skipping the model registry:** Deploying a model directly from a run artifact bypasses the staging and evaluation workflow. The registry exists to enforce the requirement that every production model was evaluated before promotion.



---

**The MLOps Internship** -- Book 4 of 9, InternshipInABook™

Dataset: CreditDefault-NG | Company: DataFlow Infrastructure, Lagos/Abuja

[internshipinabook.com](https://internshipinabook.com)


📘 **Get the complete illustrated book (all 13 weeks, DOCX + EPUB):** [https://selar.com/7m27877571](https://selar.com/7m27877571)